In [2]:
!pip install pycuda

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 33.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.8/98.8 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 kB 14.0 MB/s eta 0:00:00
  Created wheel for pycuda: filename=pycuda-2026.1-cp312-cp312-linux_x86_64.whl size=659447 sha256=fec5c2c92c8212c8ba4f5337b04ea635854682fb13bb9399d219f1cabd30f76a
  Stored in directory: /root/.cache/pip/wheels/90/2a/71/75ec0cc316cc0ff494bfffa2935e02580129cb7f859a0cfd8f
Successfully built pycuda


In [ ]:
import numpy as np
import pycuda.driver as cuda
import pycuda.autoinit
from pycuda.compiler import SourceModule
import time

vector_sum_kernel = """
__global__ void addKernel(int* result, int* a, unsigned int size) {
    int index = blockIdx.x * blockDim.x + threadIdx.x;

    extern __shared__ int sharedSum[];

    int localSum = 0;

    if (index < size) {
        localSum = a[index];
    }

    sharedSum[threadIdx.x] = localSum;
    __syncthreads();

    for (int stride = blockDim.x / 2; stride > 0; stride >>= 1) {
        if (threadIdx.x < stride) {
            sharedSum[threadIdx.x] += sharedSum[threadIdx.x + stride];
        }
        __syncthreads();
    }

    if (threadIdx.x == 0) {
        atomicAdd(result, sharedSum[0]);
    }
}
"""

mod = SourceModule(vector_sum_kernel)
vector_sum = mod.get_function("addKernel")


def vector_sum_gpu(vector):
    start_time = time.time()

    vector_gpu = cuda.mem_alloc(vector.nbytes)
    result_gpu = cuda.mem_alloc(np.int32().nbytes)

    cuda.memcpy_htod(vector_gpu, vector)
    cuda.memcpy_htod(result_gpu, np.array([0], dtype=np.int32))

    block_size = 256
    grid_size = (len(vector) + block_size - 1) // block_size

    vector_sum(
        result_gpu,
        vector_gpu,
        np.int32(len(vector)),
        block=(block_size, 1, 1),
        grid=(grid_size, 1),
        shared=block_size * np.int32().nbytes
    )

    cuda.Context.synchronize()

    result = np.empty(1, dtype=np.int32)
    cuda.memcpy_dtoh(result, result_gpu)

    end_time = time.time()

    return result[0], end_time - start_time


def vector_sum_cpu(vector):
    answer = 0
    for elem in vector:
        answer += elem
    return answer


if __name__ == "__main__":
    sizes = [
        2_000,
        4_000,
        8_000,
        16_000,
        32_000,
        64_000,
        128_000,
        256_000,
        512_000,
        1_024_000,
        2_048_000,
        4_096_000,
        8_192_000,
        8_194_000
    ]

    print("Size | CPU time | GPU time |")
    print("----------------------------")

    for vector_size in sizes:
        vector = np.random.randint(1, 10, size=vector_size, dtype=np.int32)

        start_time_cpu = time.time()
        answer_cpu = vector_sum_cpu(vector)
        time_cpu = time.time() - start_time_cpu

        answer_gpu, time_gpu = vector_sum_gpu(vector)

        if answer_cpu != answer_gpu:
            print(f"Ошибка на размере {vector_size}!")

        print(f"{vector_size} | {time_cpu:.6f} | {time_gpu:.6f}")

Size | CPU time | GPU time |
----------------------------
2000 | 0.000170 | 0.002307
4000 | 0.000348 | 0.000250
8000 | 0.000678 | 0.000199
16000 | 0.001323 | 0.000221
32000 | 0.002583 | 0.000271
64000 | 0.005194 | 0.000280
128000 | 0.010986 | 0.000364
256000 | 0.020721 | 0.000580
512000 | 0.041828 | 0.000809
1024000 | 0.100196 | 0.001436
2048000 | 0.167970 | 0.002995
4096000 | 0.342020 | 0.005057
8192000 | 0.691938 | 0.008457
8194000 | 0.711776 | 0.008347
